# Rubik AI: training DeepCubeA on Google Colab (GPU)

DeepCubeA learns to estimate **how many moves are left until the cube is solved** (a value network),
trained with *Deep Approximate Value Iteration*, and then solves cubes with **batch weighted A\***.
Its architecture is the same as the large EfficientCube model (14.7 million parameters), so the comparison is fair.

Same flow as the previous notebook: run the cells from top to bottom. Checkpoints are saved to
Google Drive, so if Colab disconnects, run cells 1–4 again and then cell 5 to resume.

## 1. Make sure the GPU is on

Menu **Runtime → Change runtime type → T4 GPU → Save**.

In [ ]:
!nvidia-smi

## 2. Connect Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/Rubik_AI_colab"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Storage folder:", DRIVE_DIR)

## 3. Get the code from GitHub

For a private repository, create a new token as before (fine-grained, only the **Rubik_AI** repository, **Contents: Read-only**).
Copy the token, **never take a screenshot of it**, and paste it into the box that appears. For a public repository, leave the box empty.

In [ ]:
import getpass, os, subprocess

REPO = "HaiqalGITuru/Rubik_AI"
WORKDIR = "/content/Rubik_AI"

if not os.path.exists(WORKDIR):
    token = getpass.getpass("Paste your GitHub token (leave empty for a public repository): ")
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    r = subprocess.run(["git", "clone", url, WORKDIR], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr)
    subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"])
    del token
else:
    print("Code already present in", WORKDIR)

os.chdir(WORKDIR)
!ls

## 4. Training settings

- Every step: 300 scrambles × 30 moves = **9,000 positions**, and for every position the target network
  scores its 12 children (108,000 evaluations per step).
- **The target network is refreshed every 500 steps**, so 50,000 steps = 100 refreshes. This matters:
  the distance estimate can only grow by about +1 per refresh, while a fully scrambled cube is
  up to about 26 quarter turns away.
- After cell 5 has run for a few minutes, check the `~... min left` figure. If the total is far above
  ~5 hours (the training time of the large EfficientCube model), stop it, lower `STEPS`, and start again
  **after deleting the checkpoint file on Drive**. Do not change `STEPS` in the middle of training.

In [ ]:
MODEL_PATH = f"{DRIVE_DIR}/deepcubea_large.pt"
STEPS = 50_000
SCRAMBLES = 300
UPDATE_EVERY = 500
print("Checkpoint:", MODEL_PATH, "| exists:", os.path.exists(MODEL_PATH))

## 5. Training

Watch the **target mean** and **max** columns: they rise slowly with every target-network refresh
until they settle around the true distances. This is DeepCubeA "learning to count the distance".

In [ ]:
!python -m scripts.train_deepcubea --device cuda --hidden1 5000 --hidden2 1000 --blocks 4 --scrambles {SCRAMBLES} --max-depth 30 --steps {STEPS} --update-every {UPDATE_EVERY} --cosine --log-every 200 --eval-every 2000 --save {MODEL_PATH} --resume

## 6. Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(MODEL_PATH.replace(".pt", "_log.csv"))
ev = log.dropna(subset=["solve_d20"])

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(log.step, log.loss, color="#E8762D"); ax[0].set_title("Loss (MSE)"); ax[0].set_xlabel("step")
ax[1].plot(log.step, log.target_mean, color="#E8762D", label="mean")
ax[1].plot(log.step, log.target_max, color="#E8762D", alpha=0.4, label="maximum")
ax[1].set_title("Distance estimate (DAVI target)"); ax[1].set_xlabel("step"); ax[1].legend(frameon=False)
for d, c in [(10, "#F2C2A0"), (15, "#EE9A62"), (20, "#E8762D")]:
    ax[2].plot(ev.step, ev[f"solve_d{d}"] * 100, "o-", color=c, ms=3, label=f"depth {d}")
ax[2].set_title("Success rate (quick evaluation)"); ax[2].set_ylabel("%"); ax[2].set_xlabel("step"); ax[2].legend(frameon=False)
for a in ax: a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/training_curves_deepcubea.png", dpi=150)
plt.show()

## 7. Export a lightweight model (~29 MB)

In [ ]:
INFER_PATH = f"{DRIVE_DIR}/deepcubea_large_infer.pt"
!python -m scripts.export_model {MODEL_PATH} {INFER_PATH}

## 8. Test on the same 100 cubes

Batch weighted A\* with weight 0.6 (the setting reported in the DeepCubeA paper) and batch sizes
N = 100, 1,000 and 10,000. Results are copied straight to Drive.

In [ ]:
import shutil
from IPython.display import Image, display

!python -m scripts.beam_experiment --model {INFER_PATH} --device cuda --solver astar --beams 100 1000 10000 --name deepcubea_large

shutil.copytree("results/deepcubea_large", f"{DRIVE_DIR}/deepcubea_large", dirs_exist_ok=True)
display(Image("results/deepcubea_large/overview.png"))

## 9. EfficientCube vs DeepCubeA (all on the same GPU)

Uses the EfficientCube results already in the repository (`results/beam_experiment_*`).

In [ ]:
!python -m scripts.compare_results beam_experiment_small_gpu:"EfficientCube small" beam_experiment_large:"EfficientCube large" deepcubea_large:"DeepCubeA large"

shutil.copytree("results/comparison", f"{DRIVE_DIR}/comparison", dirs_exist_ok=True)
display(Image("results/comparison/comparison.png"))

## 10. Bring the results to your laptop

From the **Rubik_AI_colab** folder on Google Drive, download:

- `deepcubea_large_infer.pt` → `Rubik_AI/models/`
- the folders `deepcubea_large` and `comparison` → `Rubik_AI/results/` (unzip, then **drag the folder in**; do not use "Extract All")
- `deepcubea_large_log.csv` and `training_curves_deepcubea.png` → `Rubik_AI/results/colab/`

When you are done, delete the GitHub token you used for Colab.